# Statistik mit Python

**Einführung anhand von Messdaten eines Web-Systems**

In diesem Notebook werden grundlegende Verfahren der deskriptiven Statistik mit Python angewendet. Python dient zur Aufbereitung, Auswertung und grafischen Darstellung der Daten.

## Lernziele

Nach der Bearbeitung des Notebooks können Sie

- die Vektor- und Summennotation aus Vorlesung 0 in einfachen Python-Ausdrücken wiedererkennen,
- Grundgesamtheit, Merkmale, Ausprägungen und Wertemengen an einem Datensatz erläutern,
- nominal, ordinal und metrisch skalierte Merkmale sowie diskrete und stetige Merkmale unterscheiden,
- Daten auswählen, filtern und gruppieren,
- absolute und relative Häufigkeiten sowie einen Modalwert bestimmen,
- arithmetisches Mittel, Median, unteres und oberes Quartil, Interquartilsabstand und empirische Standardabweichung berechnen und interpretieren,
- Säulendiagramm, Histogramm, Boxplot und Streudiagramm sachgerecht einsetzen,
- den Pearson-Korrelationskoeffizienten interpretieren,
- deskriptive Aussagen von kausalen Aussagen unterscheiden und
- die zufällige Schwankung von Stichprobenmittelwerten erläutern.

Eine Code-Zelle wird mit **Shift + Enter** ausgeführt.

## Fragestellungen

Wir betrachten Messdaten eines fiktiven Web-Systems. Jede Zeile des Datensatzes beschreibt einen API-Request.

1. Welche Endpoints werden häufig aufgerufen?
2. Wie unterscheiden sich die Antwortzeiten zwischen den Endpoints?
3. Wie sind die Antwortzeiten verteilt und wie stark streuen sie?
4. Besteht ein linearer Zusammenhang zwischen Payload-Größe und Antwortzeit?
5. Unterscheiden sich die Antwortzeiten bei Requests mit und ohne Cache-Hit?
6. Wie stark schwankt das arithmetische Mittel zwischen verschiedenen Zufallsstichproben?

## 1. Anknüpfung an Vorlesung 0

In Vorlesung 0 wurden Vektoren und das Summenzeichen eingeführt. Ein Vektor wird dort in der Form

\[
x=(x_1,x_2,\ldots,x_n)
\]

geschrieben. Die Reihenfolge der Elemente ist dabei wesentlich; Werte dürfen mehrfach auftreten.

Wir verwenden den Vektor

\[
x=(3,-4,11,-1,6).
\]

Die Werte eines solchen Vektors können in Python beispielsweise in einer Liste gespeichert werden.

In [ ]:
x = [3, -4, 11, -1, 6]

print("Anzahl der Elemente n:", len(x))
print("Erstes Element x_1:", x[0])
print("Summe der Elemente:", sum(x))

Python beginnt bei der Nummerierung von Listenelementen mit `0`. Daher entspricht `x[0]` dem mathematischen Element \(x_1\).

Auch Summen lassen sich unmittelbar berechnen. Zum Beispiel entsprechen die folgenden Python-Ausdrücke den Summen

\[
\sum_{i=1}^n x_i,\qquad
\sum_{i=1}^n x_i^2,\qquad
\sum_{i=1}^n (x_i-3)^2.
\]

Die genaue Python-Syntax ist an dieser Stelle nicht das Lernziel. Entscheidend ist die Verbindung zwischen mathematischer Schreibweise und Berechnung.

In [ ]:
print("Summe x_i:       ", sum(xi for xi in x))
print("Summe x_i^2:     ", sum(xi**2 for xi in x))
print("Summe (x_i-3)^2: ", sum((xi - 3)**2 for xi in x))

## 2. Untersuchungsgegenstand und Merkmale

Ein **API-Request** ist eine Anfrage eines Programms an einen Server. Ein **Endpoint** bezeichnet die angesprochene Funktion der Schnittstelle, beispielsweise `/search` oder `/checkout`. Die **Payload** ist die übertragene Datenmenge. Ein **Cache** ist ein Zwischenspeicher; bei einem **Cache-Hit** können bereits gespeicherte Daten verwendet werden.

Die Variablen im Datensatz haben folgende Bedeutung:

| Variable | Bedeutung |
|---|---|
| `request_id` | Kennung des Requests |
| `endpoint` | angesprochener API-Endpoint |
| `region` | Region des Requests |
| `hour` | Stunde des Tages |
| `cache_hit` | gibt an, ob Daten aus dem Cache verwendet wurden |
| `payload_kb` | Payload-Größe in Kilobyte |
| `response_ms` | Antwortzeit in Millisekunden |
| `status_code` | HTTP-Statuscode, z. B. 200, 404 oder 500 |

### Grundgesamtheit, Merkmale und Ausprägungen

Die **Grundgesamtheit** umfasst die für eine Untersuchung interessanten Daten liefernden Objekte. In unserem Beispiel können – abhängig von der Fragestellung – alle API-Requests des betrachteten Web-Systems als Grundgesamtheit aufgefasst werden.

Ein **Merkmal** ist eine für die Untersuchung interessante Eigenschaft eines solchen Objekts, beispielsweise der Endpoint oder die Antwortzeit.

Eine **Ausprägung** ist ein möglicher Wert eines Merkmals. Die **Wertemenge** \(W\) enthält alle möglichen Ausprägungen eines Merkmals.

Beispiele:

\[
W_{\mathrm{cache}}=\{\mathrm{False},\mathrm{True}\}
\]

und

\[
W_{\mathrm{endpoint}}=
\{\mathrm{/login},\mathrm{/search},\mathrm{/product},\mathrm{/cart},\mathrm{/checkout}\}.
\]

Die 320 Zeilen sind die vorliegenden Beobachtungen. Werden sie als Teil einer größeren Grundgesamtheit betrachtet, bilden sie eine **Stichprobe**.

### Merkmalsskalierungen sowie diskrete und stetige Merkmale

In der Vorlesung werden nominal, ordinal und metrisch skalierte Merkmale unterschieden. Zusätzlich wird betrachtet, ob ein Merkmal diskret oder stetig modelliert wird.

| Merkmal | Skalierung | Modellierung |
|---|---|---|
| `request_id` | nominal | diskret |
| `endpoint` | nominal | diskret |
| `region` | nominal | diskret |
| `cache_hit` | nominal | diskret |
| `status_code` | nominal | diskret |
| `hour` | metrisch und ordinal | diskret |
| `payload_kb` | metrisch und ordinal | quasi-stetig |
| `response_ms` | metrisch und ordinal | quasi-stetig |

`status_code` ist trotz der Zahlencodierung nominalskaliert. Mit den Zahlenwerten 200, 404 und 500 kann nicht im Sinne eines metrischen Merkmals gerechnet werden.

`payload_kb` und `response_ms` werden im Datensatz gerundet gespeichert. Wegen der großen Zahl möglicher Werte werden sie hier zweckmäßig als quasi-stetige Merkmale behandelt.

`hour` wird in ganzen Stunden erfasst. Da es sich um eine Uhrzeit und nicht um eine Zeitdauer handelt, verwenden wir dieses Merkmal im Folgenden nicht für Lage- oder Streuungsmaße.

## 3. Benötigte Python-Bibliotheken

Wir verwenden `pandas` zur Arbeit mit Tabellen, `numpy` für numerische Berechnungen und Zufallszahlen sowie `matplotlib` für grafische Darstellungen.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 20)
plt.rcParams["figure.figsize"] = (8, 4.5)

## 4. Datensatz einlesen und prüfen

In [ ]:
df = pd.read_csv("data/api_requests.csv")
df.head()

In [ ]:
print("Anzahl Zeilen und Spalten:", df.shape)
print("Spalten:", df.columns.to_list())

In [ ]:
df.info()

Vor einer statistischen Auswertung sollte geprüft werden, ob Werte fehlen. Fehlende Werte werden nicht pauschal entfernt. Für jede Auswertung ist zu entscheiden, welche Variablen benötigt werden.

In [ ]:
df.isna().sum()

Für Auswertungen der Antwortzeit benötigen wir Beobachtungen mit vorhandenem Wert von `response_ms`. Dafür wird ein eigener DataFrame angelegt. Bei Auswertungen anderer Merkmale kann weiterhin der vollständige Datensatz verwendet werden.

In einer realen Untersuchung müsste zusätzlich geklärt werden, weshalb Werte fehlen. Das Entfernen unvollständiger Beobachtungen kann zu Verzerrungen führen, wenn das Auftreten fehlender Werte systematisch ist.

In [ ]:
response_data = df.dropna(subset=["response_ms"]).copy()

print("Alle Requests:", len(df))
print("Requests mit gemessener Antwortzeit:", len(response_data))

## 5. Daten auswählen und filtern

Mit Bedingungen lassen sich Teilmengen eines DataFrames auswählen.

In [ ]:
# Eine Spalte auswählen
response_data["response_ms"].head()

In [ ]:
# Requests mit HTTP-Statuscode 500
server_errors = df[df["status_code"] == 500]
server_errors[["request_id", "endpoint", "response_ms", "status_code"]].head()

In [ ]:
# Mehrere Bedingungen verknüpfen
slow_checkout = response_data[
    (response_data["endpoint"] == "/checkout")
    & (response_data["response_ms"] > 400)
]

slow_checkout.head()

### Übung 1: Filtern

Filtern Sie alle Requests, die

- den Endpoint `/search` verwenden,
- aus `EU-Central` stammen und
- eine Antwortzeit von mehr als 250 ms haben.

Bestimmen Sie die Anzahl dieser Requests.

Überlegen Sie anschließend: Welche Aussage beschreibt nur die vorliegenden Beobachtungen, und welche Aussage wäre bereits eine Verallgemeinerung auf die Grundgesamtheit?

In [ ]:
# TODO: Filter formulieren und Anzahl bestimmen.

## 6. Häufigkeiten für ein diskretes Merkmal

Der Endpoint ist ein nominalskaliertes und damit diskretes Merkmal.

Für eine Ausprägung \(w\in W\) bezeichnet

- \(H(w)\) die **absolute Häufigkeit** und
- \(h(w)=H(w)/n\) die **relative Häufigkeit**.

Der häufigste Wert der Wertemenge heißt **Modalwert**. Es können mehrere Modalwerte auftreten.

Für die Frage nach der Häufigkeitsverteilung der Endpoints werden alle Requests betrachtet. Fehlende Antwortzeiten spielen für diese Auswertung keine Rolle.

In [ ]:
# Absolute Häufigkeiten
endpoint_counts = df["endpoint"].value_counts()
endpoint_counts

In [ ]:
# Relative Häufigkeiten in Prozent
endpoint_percent = df["endpoint"].value_counts(normalize=True) * 100
endpoint_percent.round(1)

In [ ]:
print("Modalwert:", df["endpoint"].mode().tolist())

In [ ]:
endpoint_counts.plot(kind="bar")
plt.title("Absolute Häufigkeiten der Endpoints")
plt.xlabel("Endpoint")
plt.ylabel("Absolute Häufigkeit")
plt.xticks(rotation=0)
plt.show()

Für nominalskalierte Merkmale sind Häufigkeitstabellen und Säulendiagramme geeignete Darstellungen. Ein Histogramm wird dagegen zur Darstellung der Verteilung eines metrisch skalierten Merkmals verwendet.

## 7. Metrisch skalierte Merkmale: Lage und Streuung

Für die Antwortzeit betrachten wir verschiedene Lage- und Streuungsmaße.

Das arithmetische Mittel eines Datenvektors \(x=(x_1,\ldots,x_n)\) ist

\[
\bar{x}=\frac{1}{n}\sum_{i=1}^n x_i.
\]

Damit erscheint das Summenzeichen aus Vorlesung 0 unmittelbar wieder.

In [ ]:
response_data["response_ms"].agg(
    ["count", "mean", "median", "std", "min", "max"]
).round(1)

### Quantile und Quartile

Für Quantile sind verschiedene Festlegungen möglich. In der Vorlesung wird für Datenvektoren metrischer Merkmale folgende Quantilformel verwendet.

Es sei \(k=\alpha n\).

- Ist \(k\) keine ganze Zahl, gilt \(q_x(\alpha)=x_{(\lceil k\rceil)}\).
- Ist \(k\) ganzzahlig, gilt
  \[
  q_x(\alpha)=\frac{x_{(k)}+x_{(k+1)}}{2}.
  \]

Dabei bezeichnet \(x_{(i)}\) die \(i\)-te Beobachtung der aufsteigend geordneten Daten.

Die Quantile

\[
q_x(0{,}25)\quad\text{und}\quad q_x(0{,}75)
\]

heißen **unteres Quartil** und **oberes Quartil**. Ihre Differenz

\[
IQR_x=q_x(0{,}75)-q_x(0{,}25)
\]

heißt **Interquartilsabstand**.

Damit die Berechnung in Python mit der Quantilformel der Vorlesung übereinstimmt, verwenden wir eine eigene Funktion.

In [ ]:
def empirisches_quantil(values, alpha):
    if not 0 < alpha < 1:
        raise ValueError("alpha muss zwischen 0 und 1 liegen.")

    ordered = np.sort(np.asarray(values))
    n = len(ordered)
    k = alpha * n

    if np.isclose(k, round(k)):
        k = int(round(k))
        return (ordered[k - 1] + ordered[k]) / 2

    k = int(np.ceil(k))
    return ordered[k - 1]


def boxplot_statistik(values, label=""):
    """Kennwerte für einen Boxplot nach der Quantildefinition der Vorlesung."""
    ordered = np.sort(pd.Series(values).dropna().to_numpy())

    q_unten = empirisches_quantil(ordered, 0.25)
    median = np.median(ordered)
    q_oben = empirisches_quantil(ordered, 0.75)
    iqr = q_oben - q_unten

    untere_grenze = q_unten - 1.5 * iqr
    obere_grenze = q_oben + 1.5 * iqr

    keine_ausreisser = ordered[(ordered >= untere_grenze) & (ordered <= obere_grenze)]
    ausreisser = ordered[(ordered < untere_grenze) | (ordered > obere_grenze)]

    return {
        "label": label,
        "whislo": keine_ausreisser.min(),
        "q1": q_unten,
        "med": median,
        "q3": q_oben,
        "whishi": keine_ausreisser.max(),
        "fliers": ausreisser.tolist(),
    }

In [ ]:
response_times = response_data["response_ms"]

q_unten = empirisches_quantil(response_times, 0.25)
q_oben = empirisches_quantil(response_times, 0.75)
iqr = q_oben - q_unten

print("Arithmetisches Mittel:          ", round(response_times.mean(), 1), "ms")
print("Median:                         ", round(response_times.median(), 1), "ms")
print("Empirische Standardabweichung: ", round(response_times.std(), 1), "ms")
print("Unteres Quartil q_x(0,25):     ", round(q_unten, 1), "ms")
print("Oberes Quartil q_x(0,75):      ", round(q_oben, 1), "ms")
print("Interquartilsabstand IQR_x:     ", round(iqr, 1), "ms")

`pandas.Series.std()` verwendet standardmäßig den Divisor \(n-1\). Damit entspricht die ausgegebene Standardabweichung der in der Vorlesung verwendeten empirischen Standardabweichung.

### Arithmetisches Mittel und Median

Das **arithmetische Mittel** berücksichtigt alle beobachteten Werte und reagiert deshalb empfindlich auf sehr große oder sehr kleine Beobachtungen.

Der **Median** ist ein Lagemaß auf Grundlage der geordneten Beobachtungen. Bei ungeradem Stichprobenumfang ist er die mittlere Beobachtung; bei geradem Stichprobenumfang wird das arithmetische Mittel der beiden mittleren Beobachtungen verwendet. Er ist gegenüber einzelnen Extremwerten robuster als das arithmetische Mittel.

In [ ]:
response_data.nlargest(8, "response_ms")[[
    "request_id", "endpoint", "region", "response_ms", "status_code"
]]

In [ ]:
response_data["response_ms"].plot.hist(bins=28, edgecolor="black")
plt.axvline(response_data["response_ms"].mean(), linestyle="--", label="Arithmetisches Mittel")
plt.axvline(response_data["response_ms"].median(), linestyle=":", label="Median")
plt.title("Histogramm der Antwortzeiten")
plt.xlabel("Antwortzeit [ms]")
plt.ylabel("Absolute Häufigkeit")
plt.legend()
plt.show()

Die Verteilung ist rechtsschief. Einzelne große Antwortzeiten erhöhen das arithmetische Mittel stärker als den Median. Bei schiefen Verteilungen ist es daher sinnvoll, mehrere Lage- und Streuungsmaße gemeinsam zu betrachten.

Auch die Wahl der Klasseneinteilung eines Histogramms beeinflusst den visuellen Eindruck der Verteilung. Ein Histogramm sollte deshalb nicht isoliert interpretiert werden.

In [ ]:
fig, ax = plt.subplots()
ax.bxp([boxplot_statistik(response_times, "response_ms")], showfliers=True)
ax.set_title("Boxplot der Antwortzeiten")
ax.set_ylabel("Antwortzeit [ms]")
plt.show()

### Boxplot

Der Boxplot gibt einen visuellen Eindruck von Lage und Streuung eines Datenvektors.

- Die Box beginnt beim **unteren Quartil** \(q_x(0{,}25)\) und endet beim **oberen Quartil** \(q_x(0{,}75)\).
- Die Linie innerhalb der Box kennzeichnet den Median.
- Die Länge der Box ist der Interquartilsabstand
  \[
  IQR_x=q_x(0{,}75)-q_x(0{,}25).
  \]
- Als **Ausreißer** gelten im Boxplot Datenwerte, die unterhalb von
  \[
  q_x(0{,}25)-1{,}5\cdot IQR_x
  \]
  oder oberhalb von
  \[
  q_x(0{,}75)+1{,}5\cdot IQR_x
  \]
  liegen.
- Die Whisker reichen vom kleinsten bis zum größten Wert, der nach dieser Regel nicht als Ausreißer gilt.

Die Bezeichnung als Ausreißer bedeutet nicht, dass die betreffende Beobachtung fehlerhaft sein muss.

## 8. Gruppenvergleich: Endpoint und Antwortzeit

Nun wird ein nominalskaliertes Merkmal (`endpoint`) gemeinsam mit einem metrisch skalierten Merkmal (`response_ms`) betrachtet.

In [ ]:
endpoint_stats = (
    response_data.groupby("endpoint")["response_ms"]
    .agg(["count", "mean", "median", "std"])
    .sort_values("median")
    .round(1)
)

endpoint_stats

In der Tabelle stehen `mean` für das arithmetische Mittel und `std` für die empirische Standardabweichung.

In [ ]:
endpoint_order = endpoint_stats.index.to_list()

stats = [
    boxplot_statistik(
        response_data.loc[response_data["endpoint"] == endpoint, "response_ms"],
        endpoint
    )
    for endpoint in endpoint_order
]

fig, ax = plt.subplots()
ax.bxp(stats, showfliers=True)
ax.set_title("Antwortzeiten nach Endpoint")
ax.set_xlabel("Endpoint")
ax.set_ylabel("Antwortzeit [ms]")
plt.xticks(rotation=20)
plt.show()

### Fragen zur Auswertung

- Welcher Endpoint hat den größten Median der Antwortzeit?
- Bei welchem Endpoint ist die Streuung besonders groß?
- Bei welchen Endpoints treten Ausreißer im Sinne der Boxplot-Regel auf?
- Unterscheiden sich arithmetisches Mittel und Median innerhalb der Gruppen deutlich?

Die Gruppenunterschiede sind zunächst deskriptive Befunde. Aus ihnen folgt noch keine Aussage über die Ursachen der Unterschiede.

## 9. Antwortzeit und Cache-Hit

Untersuchen Sie, ob sich die beobachteten Antwortzeiten zwischen Requests mit und ohne Cache-Hit unterscheiden.

Bestimmen Sie für beide Gruppen

- die Anzahl der Beobachtungen,
- das arithmetische Mittel und
- den Median der Antwortzeit.

In [ ]:
# TODO: Nach "cache_hit" gruppieren und "response_ms" auswerten.

### Diskussion

Ein Unterschied zwischen den beiden Gruppen zeigt einen statistischen Zusammenhang in den vorliegenden Daten. Daraus folgt nicht unmittelbar, dass der Cache-Hit die gesamte beobachtete Differenz verursacht.

Mögliche Drittvariablen sind beispielsweise Endpoint, Region, Payload-Größe oder Tageszeit. Prüfen Sie zunächst, ob Cache-Hits bei allen Endpoints gleich häufig auftreten.

In [ ]:
pd.crosstab(df["endpoint"], df["cache_hit"], normalize="index").round(2)

Die relativen Häufigkeiten unterscheiden sich zwischen den Endpoints. Der einfache Vergleich der beiden Cache-Gruppen ist daher eine deskriptive Auswertung, aber kein Nachweis einer kausalen Wirkung.

## 10. Zwei metrisch skalierte Merkmale: Streudiagramm und Korrelation

Wir untersuchen den Zusammenhang zwischen Payload-Größe und Antwortzeit. Zunächst wird ein Streudiagramm betrachtet.

In [ ]:
response_data.plot.scatter(x="payload_kb", y="response_ms", alpha=0.6)
plt.title("Payload-Größe und Antwortzeit")
plt.xlabel("Payload-Größe [kB]")
plt.ylabel("Antwortzeit [ms]")
plt.show()

In [ ]:
r_xy = response_data["payload_kb"].corr(response_data["response_ms"])
print("Pearson-Korrelationskoeffizient r_x,y:", round(r_xy, 3))

Der **Pearson-Korrelationskoeffizient**

\[
r_{x,y}=\frac{s_{x,y}}{s_xs_y}
\]

ist ein Maß für die lineare Korrelation zweier metrisch skalierter Merkmale. Es gilt

\[
-1\le r_{x,y}\le 1.
\]

- \(r_{x,y}=1\): perfekter positiver linearer Zusammenhang,
- \(r_{x,y}=-1\): perfekter negativer linearer Zusammenhang,
- \(r_{x,y}=0\): die beiden Merkmale sind im Sinne des Pearson-Korrelationskoeffizienten **linear unkorreliert**.

Für die Einteilung in „schwach“, „mittel“ oder „stark“ gibt es keine allgemein gültigen Grenzwerte.

Ein Korrelationskoeffizient nahe null schließt einen nichtlinearen Zusammenhang oder unterschiedliche Zusammenhänge in Teilgruppen nicht aus. Außerdem können einzelne extreme Beobachtungen den Korrelationskoeffizienten beeinflussen.

Aus einer Korrelation allein folgt keine kausale Beziehung.

Im vorliegenden Datensatz ist der lineare Gesamtzusammenhang zwischen Payload-Größe und Antwortzeit gering. Das Ergebnis schließt nicht aus, dass innerhalb einzelner Teilgruppen andere Zusammenhänge bestehen oder weitere Merkmale die Antwortzeit beeinflussen.

## 11. Zufällige Stichproben und Stichprobenmittelwerte

Für die folgende Simulation werden die Requests mit gemessener Antwortzeit als endliche Grundgesamtheit aufgefasst. Aus dieser Grundgesamtheit werden wiederholt einfache Zufallsstichproben ohne Zurücklegen gezogen.

Für jede Stichprobe mit Umfang \(n=30\) wird das arithmetische Mittel der Antwortzeiten berechnet.

In [ ]:
rng = np.random.default_rng(42)

sample_means = []
for _ in range(500):
    sample = response_data["response_ms"].sample(
        n=30,
        replace=False,
        random_state=int(rng.integers(1_000_000))
    )
    sample_means.append(sample.mean())

sample_means = pd.Series(sample_means)
sample_means.describe()

In [ ]:
sample_means.plot.hist(bins=25, edgecolor="black")
plt.axvline(
    response_data["response_ms"].mean(),
    linestyle="--",
    label="Arithmetisches Mittel der Grundgesamtheit"
)
plt.title("Verteilung von 500 Stichprobenmittelwerten (n = 30)")
plt.xlabel("Stichprobenmittelwert der Antwortzeit [ms]")
plt.ylabel("Absolute Häufigkeit")
plt.legend()
plt.show()

Die Stichprobenmittelwerte sind nicht identisch, obwohl alle Stichproben aus derselben Grundgesamtheit gezogen werden. Die Stichprobenmittelwerte unterliegen also einer **zufälligen Schwankung**.

In [ ]:
def repeated_means(n, repetitions=500, seed=123):
    local_rng = np.random.default_rng(seed)
    means = []
    for _ in range(repetitions):
        sample = response_data["response_ms"].sample(
            n=n,
            replace=False,
            random_state=int(local_rng.integers(1_000_000))
        )
        means.append(sample.mean())
    return pd.Series(means)

comparison = pd.DataFrame({
    "n=10": repeated_means(10),
    "n=30": repeated_means(30),
    "n=100": repeated_means(100),
})

comparison.std().round(1)

In dieser Simulation nimmt die Standardabweichung der Stichprobenmittelwerte mit zunehmendem Stichprobenumfang ab. Größere Zufallsstichproben liefern daher im Allgemeinen präzisere Schätzungen eines Mittelwerts, sofern das Stichprobenverfahren und die zugrunde liegenden Annahmen angemessen sind.

## 12. Abschlussaufgaben

Beantworten Sie die folgenden Fragen zunächst mit Python und formulieren Sie anschließend jeweils eine kurze statistische Aussage.

1. Welcher Endpoint hat den größten Median der Antwortzeit?
2. Wie groß ist der relative Anteil der Requests mit Statuscode 500?
3. Welche Region hat den größten Median der Antwortzeit?
4. Formulieren Sie zu Aufgabe 3 eine rein deskriptive Aussage, ohne eine Ursache für den Unterschied zu behaupten.

In [ ]:
# Aufgabe 1

In [ ]:
# Aufgabe 2

In [ ]:
# Aufgabe 3

### Beurteilung statistischer Aussagen

Welche der folgenden Aussagen sind durch die bisherigen Auswertungen gerechtfertigt?

A. Im beobachteten Datensatz hat `/checkout` den größten Median der Antwortzeit.

B. Aus dem Unterschied zwischen Requests mit und ohne Cache-Hit lässt sich unmittelbar bestimmen, um wie viele Millisekunden ein Cache-Hit die Antwortzeit kausal verkürzt.

C. Ein Pearson-Korrelationskoeffizient nahe null beweist, dass zwischen Payload-Größe und Antwortzeit keinerlei Zusammenhang bestehen kann.

## 13. Zusammenfassung

- Die Vektor- und Summennotation aus Vorlesung 0 lässt sich unmittelbar mit Python-Berechnungen verbinden.
- Zu Beginn einer statistischen Untersuchung werden Grundgesamtheit, Merkmale, Ausprägungen und Wertemengen festgelegt.
- Merkmale können nominal, ordinal oder metrisch skaliert sein und diskret oder stetig modelliert werden.
- Absolute und relative Häufigkeiten beschreiben die Häufigkeitsverteilung diskreter Merkmale.
- Arithmetisches Mittel und Median sind Lagemaße mit unterschiedlichen Eigenschaften.
- Empirische Standardabweichung und Interquartilsabstand sind Streuungsmaße.
- Histogramm und Boxplot veranschaulichen die Verteilung metrischer Merkmale.
- Der Pearson-Korrelationskoeffizient \(r_{x,y}\) beschreibt die lineare Korrelation zweier metrischer Merkmale.
- Aus einem beobachteten Zusammenhang folgt nicht ohne Weiteres eine kausale Beziehung.
- Arithmetische Mittel verschiedener Zufallsstichproben unterliegen zufälligen Schwankungen.